In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "./ewt_baseline/checkpoint-2352"

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
model = AutoModelForTokenClassification.from_pretrained(model_path)

label2id = model.config.label2id
id2label = model.config.id2label

print(label2id)
print(id2label)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-LOC': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}
{0: 'B-LOC', 1: 'B-ORG', 2: 'B-PER', 3: 'I-LOC', 4: 'I-ORG', 5: 'I-PER', 6: 'O'}


In [3]:
def read_tweebank_bio(path):
    sentences = []
    labels = []

    current_tokens = []
    current_labels = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line == "":
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
                continue

            if line.startswith("-DOCSTART-"):
                continue

            parts = line.split()

            token = parts[0]
            tag = parts[-1]

            # Convert MISC -> O
            if tag in ["B-MISC", "I-MISC"]:
                tag = "O"

            current_tokens.append(token)
            current_labels.append(tag)

    if current_tokens:
        sentences.append(current_tokens)
        labels.append(current_labels)

    return sentences, labels

In [4]:
tweebank_tokens, tweebank_labels = read_tweebank_bio("Tweebank test.bio")

unique_labels = sorted(set(label for sent in tweebank_labels for label in sent))

print(unique_labels)
print(tweebank_tokens[0])
print(tweebank_labels[0])

['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
['@USER1812', 'No', ',', 'I', "'m", 'not', '.', 'It', "'s", 'definitely', 'not', 'a', 'rapper', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [5]:
from datasets import Dataset

tweebank_dataset = Dataset.from_dict({
    "tokens": tweebank_tokens,
    "ner_tags": tweebank_labels
})

print(tweebank_dataset)
print(tweebank_dataset[0])

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 1201
})
{'tokens': ['@USER1812', 'No', ',', 'I', "'m", 'not', '.', 'It', "'s", 'definitely', 'not', 'a', 'rapper', '.'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [6]:
def encode_labels(example):
    example["ner_tags"] = [label2id[label] for label in example["ner_tags"]]
    return example

tweebank_dataset = tweebank_dataset.map(encode_labels)

print(tweebank_dataset[0])

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

{'tokens': ['@USER1812', 'No', ',', 'I', "'m", 'not', '.', 'It', "'s", 'definitely', 'not', 'a', 'rapper', '.'], 'ner_tags': [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]}


In [7]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [8]:
tokenized_tweebank = tweebank_dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_tweebank[0].keys())
print(tokenized_tweebank[0]["labels"])

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

dict_keys(['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])
[-100, 6, -100, -100, -100, -100, 6, 6, 6, 6, -100, 6, 6, 6, 6, -100, 6, 6, 6, 6, 6, -100]


In [11]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./crossdomain_results",
    per_device_eval_batch_size=8,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator
)

In [12]:
predictions, labels, metrics = trainer.predict(tokenized_tweebank)

print(metrics)

c:\Users\LG\anaconda3\envs\ner\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'test_loss': 0.11356417834758759, 'test_model_preparation_time': 0.0086, 'test_runtime': 50.9023, 'test_samples_per_second': 23.594, 'test_steps_per_second': 2.966}


In [13]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

predicted_ids = np.argmax(predictions, axis=2)

true_predictions = []
true_labels = []

for pred, lab in zip(predicted_ids, labels):
    current_preds = []
    current_labels = []

    for p, l in zip(pred, lab):
        if l != -100:
            current_preds.append(id2label[p])
            current_labels.append(id2label[l])

    true_predictions.append(current_preds)
    true_labels.append(current_labels)

print("Precision:", precision_score(true_labels, true_predictions))
print("Recall:", recall_score(true_labels, true_predictions))
print("F1:", f1_score(true_labels, true_predictions))

print(classification_report(true_labels, true_predictions))

Precision: 0.7325842696629213
Recall: 0.5759717314487632
F1: 0.6449060336300693
              precision    recall  f1-score   support

         LOC       0.65      0.77      0.70       111
         ORG       0.56      0.25      0.35       182
         PER       0.84      0.71      0.77       273

   micro avg       0.73      0.58      0.64       566
   macro avg       0.68      0.58      0.61       566
weighted avg       0.71      0.58      0.62       566



In [14]:
with open("tweebank_predictions.txt", "w", encoding="utf-8") as f:
    for preds in true_predictions:
        for p in preds:
            f.write(p + "\n")
        f.write("\n")

We begin Mitgation 1 (Text Normalisation)

In [15]:
import re

def normalize_token(token):
    # Replace usernames
    if token.startswith("@"):
        return "@USER"
    
    # Replace URLs
    if token.startswith("http") or token.startswith("www"):
        return "URL"
    
    # Reduce repeated characters: loooove -> loove
    token = re.sub(r"(.)\1{2,}", r"\1\1", token)
    
    # Reduce repeated punctuation: !!! -> !
    token = re.sub(r"([!?.,])\1+", r"\1", token)
    
    return token


In [16]:
normalized_tweebank_tokens = [
    [normalize_token(token) for token in sentence]
    for sentence in tweebank_tokens
]

print("Original:")
print(tweebank_tokens[0])

print("\nNormalised:")
print(normalized_tweebank_tokens[0])

Original:
['@USER1812', 'No', ',', 'I', "'m", 'not', '.', 'It', "'s", 'definitely', 'not', 'a', 'rapper', '.']

Normalised:
['@USER', 'No', ',', 'I', "'m", 'not', '.', 'It', "'s", 'definitely', 'not', 'a', 'rapper', '.']


In [17]:
normalized_tweebank_dataset = Dataset.from_dict({
    "tokens": normalized_tweebank_tokens,
    "ner_tags": tweebank_labels
})

normalized_tweebank_dataset = normalized_tweebank_dataset.map(encode_labels)

print(normalized_tweebank_dataset[0])

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

{'tokens': ['@USER', 'No', ',', 'I', "'m", 'not', '.', 'It', "'s", 'definitely', 'not', 'a', 'rapper', '.'], 'ner_tags': [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]}


In [18]:
tokenized_normalized_tweebank = normalized_tweebank_dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_normalized_tweebank[0].keys())
print(tokenized_normalized_tweebank[0]["labels"])

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

dict_keys(['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])
[-100, 6, -100, -100, 6, 6, 6, 6, -100, 6, 6, 6, 6, -100, 6, 6, 6, 6, 6, -100]


In [19]:
norm_predictions, norm_labels, norm_metrics = trainer.predict(tokenized_normalized_tweebank)

print(norm_metrics)

{'test_loss': 0.11292561143636703, 'test_model_preparation_time': 0.0086, 'test_runtime': 43.8857, 'test_samples_per_second': 27.367, 'test_steps_per_second': 3.441}


In [20]:
norm_predicted_ids = np.argmax(norm_predictions, axis=2)

norm_true_predictions = []
norm_true_labels = []

for pred, lab in zip(norm_predicted_ids, norm_labels):
    current_preds = []
    current_labels = []

    for p, l in zip(pred, lab):
        if l != -100:
            current_preds.append(id2label[p])
            current_labels.append(id2label[l])

    norm_true_predictions.append(current_preds)
    norm_true_labels.append(current_labels)

print("Normalised Precision:", precision_score(norm_true_labels, norm_true_predictions))
print("Normalised Recall:", recall_score(norm_true_labels, norm_true_predictions))
print("Normalised F1:", f1_score(norm_true_labels, norm_true_predictions))

print(classification_report(norm_true_labels, norm_true_predictions))

Normalised Precision: 0.7325842696629213
Normalised Recall: 0.5759717314487632
Normalised F1: 0.6449060336300693
              precision    recall  f1-score   support

         LOC       0.65      0.77      0.70       111
         ORG       0.55      0.26      0.35       182
         PER       0.85      0.71      0.77       273

   micro avg       0.73      0.58      0.64       566
   macro avg       0.68      0.58      0.61       566
weighted avg       0.71      0.58      0.62       566



In [30]:
changed = 0
total = 0

for original_sent, norm_sent in zip(tweebank_tokens, normalized_tweebank_tokens):
    for original, norm in zip(original_sent, norm_sent):
        total += 1
        if original != norm:
            changed += 1

print("Changed tokens:", changed)
print("Total tokens:", total)
print("Percentage changed:", changed / total * 100)

Changed tokens: 1202
Total tokens: 19095
Percentage changed: 6.294841581565855
